# 04 · Fetch analyst job ads from JobTech (Arbetsförmedlingen)

**Source:** [JobTech Historical Ads API](https://jobtechdev.se/en/components/historical-ads) — all ads published on Platsbanken. Free, no API key, CC0.

> Run this notebook **before** `03_clean_merge.ipynb`. Its output file (~40 MB) is not stored in the repository.

### Defining an "analyst ad"
Free-text search for *dataanalytiker* found only 83 ads in 2023, spread over unrelated occupations, and missed English-language ads.
Instead, ads are selected by **official occupation code**. The six occupations below were chosen after inspecting which occupations analyst ads are classified under.

In [ ]:
import json
import time

import pandas as pd
import requests

In [ ]:
URL = "https://historical.api.jobtechdev.se/search"

ANALYST_OCCUPATIONS = {
    "Dataanalytiker": "eTVL_hfE_Lyh",
    "Verksamhetsanalytiker": "X1FS_ZXE_DT2",
    "Data Warehouse specialist": "YpXF_Mob_zM8",
    "Statistiker": "5bYa_vBF_REN",
    "Matematisk statistiker": "GhhA_SVR_Nsr",
    "Marknadsanalytiker": "Eh2z_fA6_4Ls",
}
# Excluded on purpose: Systemanalytiker, Kravanalytiker (software design), Finansanalytiker (finance)

## Exploration: which occupations do "dataanalytiker" ads fall under?

`limit=0` returns no ads, only counts. `stats` asks the server to group the matches by occupation.

In [ ]:
params = {
    "q": "dataanalytiker",
    "published-after": "2023-01-01",
    "published-before": "2023-12-31",
    "limit": 0,
    "stats": "occupation-name",
    "stats.limit": 15,
}
response = requests.get(URL, params=params, timeout=300)
response.raise_for_status()
data = response.json()

print("total ads:", data["total"]["value"])
pd.DataFrame(data["stats"][0]["values"])

## Ads per occupation per year

This revealed a **taxonomy break**: *Dataanalytiker* did not exist before 2023 (0 ads 2020–2022).
Counting a single occupation would show fake growth, so the analysis always uses the **sum of all six**.

In [ ]:
rows = []
for year in range(2020, 2025):
    for name, concept_id in ANALYST_OCCUPATIONS.items():
        params = {
            "occupation-name": concept_id,
            "published-after": f"{year}-01-01",
            "published-before": f"{year}-12-31",
            "limit": 0,
        }
        response = requests.get(URL, params=params, timeout=300)
        response.raise_for_status()
        rows.append({"year": year, "occupation": name, "ads": response.json()["total"]["value"]})
        time.sleep(0.5)

by_year = pd.DataFrame(rows)
by_year.pivot(index="occupation", columns="year", values="ads")

## Download all ads

The API returns at most 100 ads per request and no more than ~2,100 per search, paged with `offset`.
One search per year stays below that limit (largest year: 2022, 1,898 ads). An `assert` stops the run if a year ever hits the cap,
instead of silently returning incomplete data.

Passing a **list** to `occupation-name` searches all six occupations at once (OR).

In [ ]:
OCCUPATION_IDS = list(ANALYST_OCCUPATIONS.values())
PAGE_SIZE = 100
API_CAP = 2100


def fetch_year(year: int) -> list:
    """Download every analyst ad published in one calendar year."""
    ads, offset = [], 0
    while True:
        params = {
            "occupation-name": OCCUPATION_IDS,
            "published-after": f"{year}-01-01T00:00:00",
            "published-before": f"{year}-12-31T23:59:59",
            "limit": PAGE_SIZE,
            "offset": offset,
        }
        response = requests.get(URL, params=params, timeout=300)
        response.raise_for_status()
        hits = response.json()["hits"]
        ads.extend(hits)

        if len(hits) < PAGE_SIZE:      # last page
            break
        offset += PAGE_SIZE
        time.sleep(0.5)
    return ads

In [ ]:
all_ads = []
for year in range(2020, 2025):
    ads = fetch_year(year)
    assert len(ads) < API_CAP, f"{year}: hit the {API_CAP}-ad limit, data may be incomplete"
    print(year, "->", len(ads), "ads")
    all_ads.extend(ads)

print("total:", len(all_ads))   # 6,938 — matches the sum of the per-occupation counts above

with open("../data_raw/jobtech_analyst_ads_2020_2024.json", "w", encoding="utf-8") as f:
    json.dump(all_ads, f, ensure_ascii=False)   # keep Swedish characters readable